# Diabetic Retinopathy Severity Classification
EfficientNet-B0 fine-tuned on the [APTOS 2019 Blindness Detection](https://www.kaggle.com/c/aptos2019-blindness-detection/data) dataset.

**Before running:** Make sure the runtime is set to GPU — Runtime → Change runtime type → T4 GPU.

## 1. Git Setup

In [ ]:
!git config --global user.name "YOUR_GITHUB_USERNAME"
!git config --global user.email "YOUR_GITHUB_EMAIL"

## 2. Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/noteesh/Diabetic-Retinopathy-Severity-Classification.git"
REPO_DIR = "Diabetic-Retinopathy-Severity-Classification"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print("Working directory:", os.getcwd())

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Kaggle API Setup

You need a `kaggle.json` API token. Get it from [kaggle.com](https://www.kaggle.com) → Account → API → Create New Token.

**Option A (recommended) — Colab Secrets:**
1. Click the key icon in the left sidebar
2. Add secret named `KAGGLE_KEY` with the full contents of your `kaggle.json`
3. Run the cell below

**Option B — Upload file manually:**
Upload `kaggle.json` using the Files panel, then skip to the dataset download cell.

In [ ]:
import json, os

try:
    from google.colab import userdata
    kaggle_creds = json.loads(userdata.get('KAGGLE_KEY'))
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    kaggle_json_path = os.path.expanduser('~/.kaggle/kaggle.json')
    with open(kaggle_json_path, 'w') as f:
        json.dump(kaggle_creds, f)
    os.chmod(kaggle_json_path, 0o600)
    print("Kaggle credentials loaded from Colab Secrets.")
except Exception:
    # Option B: kaggle.json uploaded manually to /content/
    src = '/content/kaggle.json'
    dst_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(dst_dir, exist_ok=True)
    import shutil
    shutil.copy(src, os.path.join(dst_dir, 'kaggle.json'))
    os.chmod(os.path.join(dst_dir, 'kaggle.json'), 0o600)
    print("Kaggle credentials loaded from uploaded file.")

## 5. Download APTOS 2019 Dataset

In [ ]:
!pip install -q kaggle

os.makedirs('data', exist_ok=True)

!kaggle competitions download -c aptos2019-blindness-detection -p data/
!unzip -q data/aptos2019-blindness-detection.zip -d data/

# Confirm expected files are present
print("CSV files:", [f for f in os.listdir('data') if f.endswith('.csv')])
train_imgs = 'data/train_images'
if os.path.exists(train_imgs):
    print(f"Train images: {len(os.listdir(train_imgs))} files")

## 6. Verify GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Switch runtime to T4 GPU for reasonable training speed.")

## 7. Train the Model

Adjust `--epochs` and `--batch-size` to taste. T4 GPU handles batch size 32 comfortably at 224px.

In [ ]:
os.makedirs('results', exist_ok=True)

!python src/run_improved.py \
    --csv-path data/train.csv \
    --img-dir data/train_images \
    --results-dir results \
    --epochs 20 \
    --batch-size 32 \
    --img-size 224 \
    --lr 1e-4 \
    --weight-decay 1e-4 \
    --label-smoothing 0.1 \
    --dropout 0.3 \
    --num-workers 2

## 8. Results

In [ ]:
from IPython.display import Image as IPyImage, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

result_plots = [
    ('results/improved_training_curves.png', 'Training Curves'),
    ('results/improved_confusion_matrix.png', 'Confusion Matrix'),
]

for path, title in result_plots:
    if os.path.exists(path):
        fig, ax = plt.subplots(figsize=(12, 5) if 'curves' in path else (8, 7))
        ax.imshow(mpimg.imread(path))
        ax.axis('off')
        ax.set_title(title, fontsize=14, pad=10)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Not found: {path}")

## 9. Save Results to Google Drive (optional)

In [ ]:
# Uncomment to mount Drive and copy results

# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree('results', '/content/drive/MyDrive/DR_results', dirs_exist_ok=True)
# print("Results saved to Google Drive.")